# Laboratori 07: Diferencat e fundme dhe simulimi i valëve

**Lënda:** Simulim dhe Metoda Numerike  
**Kohëzgjatja:** 120 minuta  
**Mjedisi kryesor:** Python/NumPy; pjesa e dytë përmban zbatimin thelbësor në GNU Octave.

**Pyetja udhëheqëse:** Si shndërrohet ekuacioni valor në një algoritëm të qëndrueshëm që ruan kuptimin e përhapjes?

Ky laborator nuk duhet lexuar si një listë komandash. Çdo pjesë nis nga modeli dhe nga një llogaritje që mund të kryhet me dorë; kodi përdoret më pas për të zgjeruar llogaritjen, për ta verifikuar dhe për të eksploruar regjime që nuk trajtohen dot me disa hapa algjebrikë. Ritmi i punës është

$$\text{pyetje}\longrightarrow\text{model}\longrightarrow\text{derivim}\longrightarrow\text{algoritëm}\longrightarrow\text{kod}\longrightarrow\text{kontroll}.$$


## Rezultatet e të nxënit

Në fund të seancës studenti duhet të jetë në gjendje të:

- nxjerrë skemën qendrore të ekuacionit valor;
- zbatojë kushtet fillestare dhe kufitare;
- llogarisë me dorë koston në kohë dhe kujtesë;
- nxjerrë dhe testojë kushtin CFL;
- shpjegojë dispersimin numerik;
- përdorë energjinë diskrete si kontroll;
- paraqesë evolucionin me grafik dinamik.


## Organizimi i seancës

| Koha | Veprimtaria | Produkti |
|---:|---|---|
| 0–15 min | Pyetja fizike dhe llogaritja e parë me dorë | Parashikim i shkruar |
| 15–40 min | Derivimi dhe algoritmi minimal | Formula dhe pseudokodi |
| 40–65 min | Zbatimi i udhëhequr në Python | Rezultat i verifikuar |
| 65–82 min | Kompleksiteti: formulë dhe matje | $T(n)$ dhe grafik kohor |
| 82–100 min | Qëndrueshmëria dhe diagnostika | Krahasim i dy rasteve |
| 100–112 min | Shembulli i plotë dhe punishte | Eksperiment i modifikuar |
| 112–120 min | GNU Octave dhe ekzekutimi lokal | Skript i riprodhueshëm |

Seksioni i shembullit të plotë është i detyrueshëm. Në punishte zgjidhen vetëm një ose dy ndryshime parametrash gjatë orës; pjesa tjetër mbetet për punë të pavarur.


## Si duhet lexuar ky laborator

Kalimi nga ODE-ja te PDE-ja shton hapësirën si drejtim diskretizimi. Një zgjedhje e rrjetës tani prek njëkohësisht rezolucionin hapësinor, hapin kohor të lejuar dhe koston totale. Para kodit vizatoni stencil-in: pika $(j,n+1)$ lidhet me $(j-1,n)$, $(j,n)$, $(j+1,n)$ dhe $(j,n-1)$. Ky vizatim e bën formulën dhe varësitë e të dhënave të dukshme.

Kushtet kufitare janë pjesë e modelit fizik. Një skaj i fiksuar, një skaj i lirë dhe një kufi absorbues prodhojnë reflektime të ndryshme. Vendosja e vlerës zero në skaje nuk është thjesht një rresht teknik; ajo përfaqëson një zgjedhje eksperimentale.


## 1. Modeli fizik

Për një tel homogjen me zhvendosje të vogël tërthore $u(x,t)$,

$$\frac{\partial^2u}{\partial t^2}=c^2\frac{\partial^2u}{\partial x^2},$$

ku $c=\sqrt{T/\mu}$ është shpejtësia e valës, $T$ tensioni dhe $\mu$ masa lineare. Për skaje të fiksuara,

$$u(0,t)=u(L,t)=0.$$

Duhet dhënë edhe profili fillestar $u(x,0)=f(x)$ dhe shpejtësia fillestare $u_t(x,0)=g(x)$. Pa këto të dhëna problemi nuk është i plotë.


## 2. Derivimi i skemës

Në rrjetën $x_j=j\Delta x$, $t^n=n\Delta t$, përdorim diferencat qendrore:

$$u_{tt}(x_j,t^n)\approx\frac{u_j^{n+1}-2u_j^n+u_j^{n-1}}{\Delta t^2},$$

$$u_{xx}(x_j,t^n)\approx\frac{u_{j+1}^n-2u_j^n+u_{j-1}^n}{\Delta x^2}.$$

Barazimi dhe zgjidhja për nivelin e ri japin

$$u_j^{n+1}=2u_j^n-u_j^{n-1}
+r^2(u_{j+1}^n-2u_j^n+u_{j-1}^n),$$

ku

$$r=\frac{c\Delta t}{\Delta x}$$

është numri Courant. Skema është e rendit të dytë në hapësirë dhe kohë.


### 2.1 Një hap me dorë

Marrim pesë pika të brendshme, $r=0.5$, nivelin e mëparshëm zero dhe profilin aktual

$$u^n=(0,0,1,0,0).$$

Në pikën qendrore,

$$u_2^{n+1}=2(1)-0+0.25(0-2+0)=1.5.$$

Në pikat fqinje,

$$u_1^{n+1}=0+0.25(1)=0.25,\qquad u_3^{n+1}=0.25.$$

Ky hap tregon se skema është lokale: çdo pikë merr informacion vetëm nga fqinjët në nivelin e mëparshëm.


# Pjesa I — Python

## Përgatitja e mjedisit

Importet vendosen në një qelizë të vetme që eksperimenti të jetë i lexueshëm dhe i riprodhueshëm. Fara e gjeneratorit deklarohet edhe kur pjesa kryesore është deterministe; kështu çdo shtesë stokastike mund të përsëritet.


In [ ]:
import math
from time import perf_counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)  # realizim i riprodhueshëm
np.set_printoptions(precision=6, suppress=True)


## 3. Kushti i parë kohor

Skema kërkon dy nivele, $u^0$ dhe $u^1$. Nga Taylor-i,

$$u_j^1=u_j^0+\Delta t\,g_j+
\frac{r^2}{2}(u_{j+1}^0-2u_j^0+u_{j-1}^0)+O(\Delta t^3).$$

Përdorimi i thjeshtë $u^1=u^0$ kur $g=0$ humb termin e nxitimit dhe ul cilësinë e fillimit.


In [ ]:
def profil_gaussian(x,x0=0.35,sigma=0.06):
    return np.exp(-0.5*((x-x0)/sigma)**2)

def vala_1d(L=1.0,c=1.0,Nx=201,T=1.5,r=0.9,ruaj_cdo=10):
    x=np.linspace(0,L,Nx); dx=x[1]-x[0]; dt=r*dx/c
    Nt=int(np.ceil(T/dt)); dt=T/Nt; r_real=c*dt/dx
    u0=profil_gaussian(x); u0[[0,-1]]=0.0
    g0=np.zeros_like(x)
    u1=u0.copy()
    u1[1:-1]=u0[1:-1]+dt*g0[1:-1]+0.5*r_real**2*(u0[2:]-2*u0[1:-1]+u0[:-2])
    u1[[0,-1]]=0.0
    kohët=[0.0]; profile=[u0.copy()]
    um,u=u0,u1
    for n in range(1,Nt):
        up=np.zeros_like(u)
        up[1:-1]=2*u[1:-1]-um[1:-1]+r_real**2*(u[2:]-2*u[1:-1]+u[:-2])
        if n%ruaj_cdo==0:
            kohët.append((n+1)*dt); profile.append(up.copy())
        um,u=u,up
    return x,np.array(kohët),np.array(profile),dt,r_real

x,tk,U,dt,r=vala_1d()
print('dt=',dt,'r=',r,'profile të ruajtura=',len(tk))


In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
for indeks in np.linspace(0,len(tk)-1,5,dtype=int):
    ax.plot(x,U[indeks],label=f't={tk[indeks]:.2f}')
ax.set(xlabel='x [m]',ylabel='u(x,t)',title='Përhapja dhe reflektimi')
ax.grid(alpha=.3); ax.legend(); plt.show()


## Si do ta llogarisim kompleksitetin

Kompleksiteti nuk përcaktohet fillimisht me kronometër. Së pari zgjedhim madhësinë e hyrjes $n$, identifikojmë veprimin dominues dhe nxjerrim formulën $T(n)$. Vetëm pasi kemi një parashikim matematikor kryejmë kohëmatjen.

Për një cikël që kryen $c$ veprime në secilin prej $n$ hapave,

$$T(n)=cn+d\in\Theta(n).$$

Për dy cikle të plota të ndërfutura,

$$T(n)=\sum_{i=1}^{n}\sum_{j=1}^{n}c=cn^2\in\Theta(n^2).$$

Koha e matur modelohet si $t(n)\approx Cn^p$. Në koordinata log–log,

$$\log t=\log C+p\log n,$$

prandaj pjerrësia $p$ krahasohet me rendin e parashikuar. Matja mund të ndikohet nga kostot fikse, interpretuesi dhe memoria; ajo është kontroll empirik, jo zëvendësim i derivimit.


### 4.1 Kompleksiteti i skemës me dorë

Ka $N_x-2$ pika të brendshme dhe $N_t$ hapa kohorë. Në çdo pikë kryhet një numër konstant operacionesh, prandaj

$$T(N_x,N_t)=a(N_x-2)N_t+bN_t\in\Theta(N_xN_t).$$

Nëse ruajmë gjithë fushën, kujtesa është $\Theta(N_xN_t)$. Për përparimin e skemës duhen vetëm tre nivele dhe kujtesa është $\Theta(N_x)$. Nëse $\Delta t$ lidhet me $\Delta x$ nga CFL dhe $T,L$ fiksohen, $N_t\propto N_x$; kështu kostoja totale në një përmasë rritet si $\Theta(N_x^2)$.


In [ ]:
Nxs=np.array([51,101,201,401,801]); kohet=[]
for Nx in Nxs:
    t0=perf_counter(); vala_1d(Nx=int(Nx),T=0.3,ruaj_cdo=10_000); kohet.append(perf_counter()-t0)
pjerr=np.polyfit(np.log(Nxs),np.log(kohet),1)[0]
print('pjerrësia ndaj Nx kur Nt∝Nx:',pjerr,'; pritja asimptotike ≈2')


## Si do ta analizojmë qëndrueshmërinë

**Kushtëzimi** i përket problemit: sa ndryshon zgjidhja e saktë kur të dhënat perturbohen pak? **Qëndrueshmëria** i përket algoritmit: sa gabim shton procedura e llogaritjes? Në trajtë skematike,

$$\text{gabimi përpara}\lesssim
\text{numri i kushtëzimit}\times\text{gabimi prapa}.$$

Në secilin laborator do të përdorim një provë konkrete: mbetje, ligj ruajtjeje, kufi stabiliteti, rezultat analitik ose përsëritje me hap më të vogël. Fjalia “grafiku duket mirë” nuk është provë numerike.


### 5.1 Analiza von Neumann dhe kushti CFL

Vendosim një mod Fourier $u_j^n=G^ne^{ikj\Delta x}$. Pas zëvendësimit në skemë merret një ekuacion për $G$. Kushti që modulet të mos rriten kërkon

$$r=\frac{c\Delta t}{\Delta x}\leq1.$$

Interpretimi fizik është i drejtpërdrejtë: në një hap kohor informacioni nuk duhet të kalojë më shumë se një qelizë. Për $r>1$, amplitudat rriten artificialisht.


In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
for rr in [0.5,0.9,1.01]:
    try:
        x,tt,UU,_,rreal=vala_1d(Nx=201,T=.7,r=rr,ruaj_cdo=5)
        maxu=np.max(np.abs(UU),axis=1)
        ax.semilogy(tt,maxu,label=f'r={rreal:.3f}')
    except FloatingPointError:
        pass
ax.set(xlabel='t [s]',ylabel=r'$\max|u|$',title='Prova e CFL-së')
ax.grid(alpha=.3); ax.legend(); plt.show()


### 5.2 Dispersimi numerik

Për një valë diskrete $e^{i(kj\Delta x-\omega n\Delta t)}$ merret

$$\sin^2\left(\frac{\omega\Delta t}{2}\right)
=r^2\sin^2\left(\frac{k\Delta x}{2}\right).$$

Pra shpejtësia fazore numerike $\omega/k$ varet nga $k$. Valët me gjatësi të vogël valore përhapen me shpejtësi të ndryshme nga $c$. Qëndrueshmëria $r\leq1$ nuk e eliminon dispersimin; ajo vetëm pengon rritjen e pakufizuar.


In [ ]:
kdx=np.linspace(1e-6,np.pi,400)
fig,ax=plt.subplots(figsize=(7,4))
for rr in [0.5,0.8,1.0]:
    omega_dt=2*np.arcsin(rr*np.sin(kdx/2))
    cp_over_c=(omega_dt/kdx)/rr
    ax.plot(kdx/np.pi,cp_over_c,label=f'r={rr}')
ax.set(xlabel=r'$k\Delta x/\pi$',ylabel=r'$c_{faza,num}/c$',title='Dispersimi numerik')
ax.grid(alpha=.3); ax.legend(); plt.show()


## 6. Energjia diskrete si kontroll

Në sistemin pa humbje, energjia e vazhdueshme

$$E=\frac12\int_0^L\left[u_t^2+c^2u_x^2\right]dx$$

ruhet. Një përafrim diskret përdor diferenca për $u_t$ dhe $u_x$. Luhatje të vogla janë të pritshme; rritje eksponenciale tregon paqëndrueshmëri ose gabim implementimi.


In [ ]:
def energji_nga_profile(U,dt,dx,c=1.0):
    E=[]
    for n in range(1,len(U)):
        ut=(U[n]-U[n-1])/dt
        ux=np.diff(U[n])/dx
        E.append(.5*dx*np.sum(ut**2)+.5*c*c*dx*np.sum(ux**2))
    return np.array(E)

# Ruajmë çdo hap që dt në formulën e energjisë të jetë i saktë.
x,tt,UU,dt,r=vala_1d(Nx=201,T=.8,r=.9,ruaj_cdo=1)
E=energji_nga_profile(UU,dt,x[1]-x[0])
print('devijimi maksimal relativ i energjisë=',np.max(np.abs(E/E[0]-1)))


## 7. Shembulli i plotë dhe punishte dinamike

Eksperimenti i detyrueshëm përdor një impuls Gaussian, skaje të fiksuara, $r=0.9$ dhe kontroll energjie. Në punishte ndryshoni vetëm njërën nga: gjerësia e impulsit, vendi fillestar, $r$, numri i qelizave ose kushti kufitar. Shpjegoni ndryshimin fizik dhe numerik veçmas.


In [ ]:
def animim_vale(x,t,U,hap=2,DINAMIK=False):
    fig,ax=plt.subplots(figsize=(7,4))
    if DINAMIK:
        for n in range(0,len(t),hap):
            ax.clear(); ax.plot(x,U[n]); ax.set_ylim(-1.1,1.1)
            ax.set(xlabel='x [m]',ylabel='u',title=f't={t[n]:.3f} s'); ax.grid(alpha=.3)
            plt.pause(.03)
    else:
        for n in np.linspace(0,len(t)-1,6,dtype=int): ax.plot(x,U[n],label=f'{t[n]:.2f}s')
        ax.set(xlabel='x [m]',ylabel='u'); ax.grid(alpha=.3); ax.legend(ncol=2)
    plt.show()

animim_vale(x,tt,UU,DINAMIK=False)


### Domeni numerik i varësisë

Pas një hapi, $u_j^{n+1}$ përdor vetëm fqinjët $j-1,j,j+1$. Pas $n$ hapash, ai mund të varet nga pikat brenda $n\Delta x$. Informacioni fizik përshkon largësinë $cn\Delta t$. Që domeni fizik të përfshihet në domenin numerik kërkohet $c\Delta t\leq\Delta x$, pra kushti CFL.


In [ ]:
# Tabelë e projektimit të rrjetës para simulimit.
for Nx in [51,101,201,401]:
    dx=1/(Nx-1); dt=.9*dx; Nt=int(np.ceil(1/dt));
    print(f'Nx={Nx:3d}, dx={dx:.5f}, dt≈{dt:.5f}, Nt≈{Nt:4d}, punë≈{Nx*Nt:8d}')


### Verifikimi me një mod normal

Për skaje të fiksuara, $u(x,t)=\sin(\pi x/L)\cos(\pi ct/L)$ është zgjidhje analitike. Ajo është provë më e fortë se një impuls Gaussian sepse lejon matjen e gabimit në çdo pikë dhe rendin e konvergjencës. Gjatë rafinimit duhet mbajtur $r$ konstant që të dy hapat të zvogëlohen në mënyrë të kontrolluar.


In [ ]:
def gabim_modi(Nx,T=.37,r=.8):
    L=c=1.; x=np.linspace(0,L,Nx); dx=x[1]-x[0]; dt=r*dx/c
    Nt=int(np.ceil(T/dt)); dt=T/Nt; rr=c*dt/dx
    um=np.sin(np.pi*x); u=um+.5*rr**2*(np.roll(um,-1)-2*um+np.roll(um,1)); u[[0,-1]]=0
    for _ in range(1,Nt):
        up=np.zeros_like(u); up[1:-1]=2*u[1:-1]-um[1:-1]+rr**2*(u[2:]-2*u[1:-1]+u[:-2]); um,u=u,up
    exact=np.sin(np.pi*x)*np.cos(np.pi*T)
    return dx,np.max(np.abs(u-exact))
vals=[gabim_modi(n) for n in [51,101,201,401]]
print(vals,'rendet=',np.log2(np.array([v[1] for v in vals[:-1]])/np.array([v[1] for v in vals[1:]])))


### Si ndahet gabimi në një simulim vale

Amplituda, faza dhe shpejtësia e paketës duhen kontrolluar veçmas. Një valë mund të ruajë amplitudën, por të mbërrijë në kohën e gabuar për shkak të dispersimit. Energjia mund të mbetet e kufizuar, ndërsa forma të shtrembërohet. Për këtë arsye raporti duhet të përmbajë të paktën një normë gabimi, një kontroll energjie dhe një diskutim të fazës.


In [ ]:
# Kontroll i kushtit kufitar dhe i amplitudës së ruajtur në të dhënat ekzistuese.
print('max kufiri majtas=',np.max(np.abs(UU[:,0])))
print('max kufiri djathtas=',np.max(np.abs(UU[:,-1])))
print('amplituda globale=',np.max(np.abs(UU)))
assert np.allclose(UU[:,[0,-1]],0)


### Pyetje për vetëkontroll

1. Pse rafinimi i hapësirës detyron rafinimin e kohës?
2. Si del kostoja $O(N_x^2)$ kur intervalet fizikë fiksohen?
3. Pse $r<1$ nuk garanton mungesë të dispersimit?
4. Çfarë ndryshimi fizik sjell një kusht kufitar i lirë?


# Pjesa II — GNU Octave

## 8. Skema valore

```octave
L=1; c=1; Nx=201; dx=L/(Nx-1); r=0.9; dt=r*dx/c; T=1.0;
Nt=ceil(T/dt); dt=T/Nt; r=c*dt/dx; x=linspace(0,L,Nx);
um=exp(-0.5*((x-0.35)/0.06).^2); um([1,end])=0;
u=um; u(2:end-1)=um(2:end-1)+0.5*r^2*(um(3:end)-2*um(2:end-1)+um(1:end-2));

for n=2:Nt
    up=zeros(size(u));
    up(2:end-1)=2*u(2:end-1)-um(2:end-1)+r^2*(u(3:end)-2*u(2:end-1)+u(1:end-2));
    if mod(n,10)==0
        plot(x,up); axis([0,L,-1.1,1.1]); grid on; drawnow;
    end
    um=u; u=up;
end
fprintf('Courant r = %.6f\n',r);
```

Provoni $r=1.01$ vetëm pasi të keni ruajtur rastin referues. Ndaleni eksperimentin nëse amplituda rritet pa kontroll.


## Ekzekutimi jashtë notebook-ut

Një notebook e ndihmon shpjegimin, por studenti duhet të dijë të ekzekutojë edhe një program të zakonshëm. Qeliza vijuese krijon një skript minimal në `scripts_local/`. Ai përmban eksperimentin qendror dhe mund të ndryshohet me një editor teksti.


In [ ]:
script_dir = Path('scripts_local')
script_dir.mkdir(exist_ok=True)
script_text = "import numpy as np\nL=c=1.; Nx=201; dx=L/(Nx-1); r=.9; dt=r*dx/c; T=1.; Nt=int(np.ceil(T/dt)); dt=T/Nt; r=c*dt/dx\nx=np.linspace(0,L,Nx); um=np.exp(-.5*((x-.35)/.06)**2); um[[0,-1]]=0\nu=um.copy(); u[1:-1]=um[1:-1]+.5*r*r*(um[2:]-2*um[1:-1]+um[:-2])\nfor _ in range(1,Nt):\n up=np.zeros_like(u); up[1:-1]=2*u[1:-1]-um[1:-1]+r*r*(u[2:]-2*u[1:-1]+u[:-2]); um,u=u,up\nprint('r=',r,'max|u|=',np.max(np.abs(u)))\n"
script_path = script_dir / 'laboratori_07_vala_1d.py'
script_path.write_text(script_text, encoding='utf-8')
print('U krijua:', script_path)


Ekzekutimi në terminal:

```bash
python3 scripts_local/laboratori_07_vala_1d.py
```

Mos raportoni vetëm se programi u ekzekutua. Shënoni komandën, një dalje numerike, një kontroll dhe një përfundim.

## Raporti i shkurtër dhe kontrolli përfundimtar

Raporti duhet të përmbajë pyetjen, modelin, supozimet, derivimin kryesor, analizën e kompleksitetit, provën e qëndrueshmërisë, figurën me boshte e njësi dhe interpretimin fizik.

- [ ] Nxora skemën nga diferencat qendrore.
- [ ] Llogarita $\Theta(N_xN_t)$ dhe kujtesën.
- [ ] Testova kufirin CFL.
- [ ] Dallova qëndrueshmërinë nga dispersimi.
- [ ] Ekzekutova skriptin Python nga terminali.
- [ ] Identifikova fragmentin përkatës në GNU Octave.
